# `long` / `wide` — Advanced Reference

`long` melts all numeric columns to rows. `wide` pivots a column's values into headers.

| Method | Key params | Purpose |
|---|---|---|
| `.long(col='variable', value='value')` | `col`, `value` | wide → long (melt) |
| `.wide(col='variable', value='value', aggfunc=None)` | `col`, `value`, `aggfunc` | long → wide (pivot) |

They are designed to work together as a pair, but each is also useful standalone.

---

In [25]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
healthexp = pt.sample_data['healthexp']
flights = pt.sample_data['flights']

## `long` — melt all numeric columns to rows

In [26]:
# Melt all numeric columns to rows — default col/value names
penguins.long().head(10)


,species,island,sex,variable,value
0,Adelie,Torgersen,Male,bill_length_mm,39.1
1,Adelie,Torgersen,Female,bill_length_mm,39.5
2,Adelie,Torgersen,Female,bill_length_mm,40.3
3,Adelie,Torgersen,None,bill_length_mm,NaN
4,Adelie,Torgersen,Female,bill_length_mm,36.7
5,Adelie,Torgersen,Male,bill_length_mm,39.3
6,Adelie,Torgersen,Female,bill_length_mm,38.9
7,Adelie,Torgersen,Male,bill_length_mm,39.2
8,Adelie,Torgersen,None,bill_length_mm,34.1
9,Adelie,Torgersen,None,bill_length_mm,42.0


In [27]:
# Custom column names for the variable and value columns
penguins.long(col='metric', value='reading').head(10)


,species,island,sex,metric,reading
0,Adelie,Torgersen,Male,bill_length_mm,39.1
1,Adelie,Torgersen,Female,bill_length_mm,39.5
2,Adelie,Torgersen,Female,bill_length_mm,40.3
3,Adelie,Torgersen,None,bill_length_mm,NaN
4,Adelie,Torgersen,Female,bill_length_mm,36.7
5,Adelie,Torgersen,Male,bill_length_mm,39.3
6,Adelie,Torgersen,Female,bill_length_mm,38.9
7,Adelie,Torgersen,Male,bill_length_mm,39.2
8,Adelie,Torgersen,None,bill_length_mm,34.1
9,Adelie,Torgersen,None,bill_length_mm,42.0


In [32]:
# Only categorical columns survive as id columns — numerics all get melted
# select narrows which numerics get melted
(penguins
 .select(['species', 'island', 'bill_length_mm', 'bill_depth_mm'])
 .long(col='metric', value='reading')
 .head(12)
)


,species,island,metric,reading
0,Adelie,Torgersen,bill_length_mm,39.1
1,Adelie,Torgersen,bill_length_mm,39.5
2,Adelie,Torgersen,bill_length_mm,40.3
3,Adelie,Torgersen,bill_length_mm,NaN
4,Adelie,Torgersen,bill_length_mm,36.7
5,Adelie,Torgersen,bill_length_mm,39.3
6,Adelie,Torgersen,bill_length_mm,38.9
7,Adelie,Torgersen,bill_length_mm,39.2
8,Adelie,Torgersen,bill_length_mm,34.1
9,Adelie,Torgersen,bill_length_mm,42.0


## `wide` — pivot a column's values into headers

In [29]:
# flights has one row per year+month — no aggfunc needed
flights.wide(col='month', value='passengers').head()


,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


In [30]:
# Select only the columns needed so island becomes the clean index
(penguins
 .select(['island', 'species', 'body_mass_g'])
 .wide(col='species', value='body_mass_g', aggfunc='mean')
)


,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


## Round-trip: `long` → transform → `wide`
A common pattern: go long to apply uniform operations across all metrics, then come back wide.

In [31]:
# Select only needed columns → long → wide: yields mean bill metrics per island
(penguins
 .select(['island', 'bill_length_mm', 'bill_depth_mm'])
 .long(col='metric', value='reading')
 .wide(col='metric', value='reading', aggfunc='mean')
)


,island,bill_depth_mm,bill_length_mm
0,Biscoe,15.874850,45.257485
1,Dream,18.344355,44.167742
2,Torgersen,18.429412,38.950980
